# mapToUnits — O2 version

Parameterized port of [mapToUnits.ipynb](mapToUnits.ipynb) for O2. Maps Neuropixels-sorted units to CCF coordinates using HERBS `probe_*.pkl` exports. Includes a batch runner at the end that auto-discovers sessions.

## Single session — edit CONFIG below

In [ ]:
# --- CONFIG: edit these for the session you want to process ---
from pathlib import Path

SESSION_LABEL = "20250305_M335_g0"          # just a tag for output filenames
MAT_DIR       = Path("/n/files/Neurobio/MICROSCOPE/Tom/CODE/LC code/processed mat + csv files/06August")
MAT_X         = MAT_DIR / "xpos_06August_sess18.mat"
MAT_Y         = MAT_DIR / "ypos_06August_sess18.mat"
PKL_DIR       = Path("/n/files/Neurobio/MICROSCOPE/Tom/CODE/LC code/processed python files/Jul30 herbs to ccf/Sess 18 20250305_M335_g0 L hand L record")
PKL_FILES     = [
    "probe_1_R_cy5.pkl",
    "probe_2_R_cy5.pkl",
    "probe_3_R_cy5.pkl",
    "probe_4_R_cy5.pkl",
]
HEMISPHERE_BACKWARDS = False   # True if shanks were put in the wrong hemisphere in HERBS
OUT_DIR = PKL_DIR              # where to save unit_sites.npy / unit_3d_coords.npy / unit_voxels.npy (+ .mat)

# Voxel-space constants (Allen CCF 10 µm)
CCF_WIDTH_X, CCF_WIDTH_Y, CCF_WIDTH_Z = 1140, 1320, 800
CCF_RESOLUTION = 0.1           # 10 µm per voxel


Load xpos / ypos and chdir to PKL_DIR

In [ ]:
import os, sys
import numpy as np
from scipy.io import loadmat

# NumPy-2 pickle compat: alias numpy._core -> numpy.core on NumPy-1
try:
    import numpy._core  # noqa: F401
except ModuleNotFoundError:
    import numpy.core as _np_core
    sys.modules['numpy._core'] = _np_core
    sys.modules.setdefault('numpy._core.multiarray', _np_core.multiarray)
    sys.modules.setdefault('numpy._core._multiarray_umath', _np_core.multiarray)
    sys.modules.setdefault('numpy._core.umath', _np_core.umath)

xpos = loadmat(str(MAT_X))['xpos'].flatten()
ypos = loadmat(str(MAT_Y))['ypos'].flatten()

os.chdir(str(PKL_DIR))
print(f"xpos n={len(xpos)}, ypos n={len(ypos)}, cwd={os.getcwd()}")


Load pkl files

In [ ]:
import pickle

class _NPCoreFixUnpickler(pickle.Unpickler):
    """Fallback: rewrite module path numpy._core -> numpy.core on the fly."""
    def find_class(self, module, name):
        if module.startswith("numpy._core"):
            module = module.replace("numpy._core", "numpy.core", 1)
        return super().find_class(module, name)

REL_KEY_CANDIDATES = (
    "sites_loc_relative", "site_loc_relative",
    "sites_loc_rel", "site_loc_rel", "rel_sites_loc",
)

def _get_rel(d):
    for k in REL_KEY_CANDIDATES:
        if k in d:
            return d[k], k
    return None, None

data_site_locs, data_rel_site_locs, data_voxels, data_ccf_sites = [], [], [], []
for i, fname in enumerate(PKL_FILES):
    with open(fname, 'rb') as f:
        try:
            loaded_data = pickle.load(f)
        except ModuleNotFoundError:
            f.seek(0)
            loaded_data = _NPCoreFixUnpickler(f).load()
    d = loaded_data['data']

    rel, used_key = _get_rel(d)
    if rel is None:
        raise KeyError(f"{fname}: no sites_loc_relative-like key; available: {sorted(d.keys())[:10]}")
    if used_key != 'sites_loc_relative':
        print(f"  [{fname}] using '{used_key}' for sites_loc_relative")

    data_site_locs.append(d['sites_loc_b'])
    data_rel_site_locs.append(rel)
    data_voxels.append(d['sites_vox'])
    data_ccf_sites.append(d['sites_label'])
    print(f"Appended to data_site_locs[{i}] from file: {fname}")


Flip CCF X if hemispheres were swapped in HERBS

In [ ]:
if HEMISPHERE_BACKWARDS:
    print("flip")
    for shank in data_site_locs:
        for col in shank:
            col[:, 0] = -col[:, 0]
    for shank in data_voxels:
        for col in shank:
            col[:, 0] = CCF_WIDTH_X - col[:, 0]


3D preview of probe site locations

In [ ]:
import plotly.graph_objs as go
import plotly.offline as pyo

fig_plotly = go.Figure()
for probe_idx, probe in enumerate(data_site_locs):
    for col_idx, arr in enumerate(probe):
        fig_plotly.add_trace(go.Scatter3d(
            x=arr[:, 0], y=arr[:, 1], z=arr[:, 2],
            mode='markers', marker=dict(size=3),
            name=f'probe {probe_idx+1} col {col_idx+1}',
        ))
fig_plotly.update_layout(
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
    title='3D positions from all entries in data_site_locs'
)
pyo.plot(fig_plotly, filename=str(OUT_DIR / 'probe_positions_3d.html'), auto_open=False)
# Always label the probes as UP in HERBS. Never left or right.


Normalise `sites_loc_relative`: flip y so lowest is 0, space out shanks on x

In [ ]:
for shank in range(4):
    for col in range(2):
        # make bottom (y) zero
        data_rel_site_locs[shank][col][:, 0] -= np.min(data_rel_site_locs[shank][col][:, 0])
        data_rel_site_locs[shank][col][:, 0] = (
            data_rel_site_locs[shank][col][:, 0] - np.max(data_rel_site_locs[shank][col][:, 0])
        ) * -1

        # space out shanks
        x_vals = data_rel_site_locs[shank][col][:, 1]
        x_vals[x_vals == 16] = 32 + 250 * shank
        x_vals[x_vals == -8] = 0  + 250 * shank
        data_rel_site_locs[shank][col][:, 1] = x_vals


Preview raw `xpos`, `ypos`

In [ ]:
import matplotlib.pyplot as plt

# Older xpos files can overflow; cast to signed int if needed (disabled by default)
if False:
    xpos = xpos.astype(np.int32)

fig = plt.figure(figsize=(5, 10))
plt.scatter(xpos, ypos)
plt.show()
print(np.min(xpos), np.max(xpos), np.min(ypos), np.max(ypos))


Map each unit to nearest CCF site

In [ ]:
from math import floor

unit_3d_coords = np.zeros((len(xpos), 3))
unit_voxels    = np.zeros((len(xpos), 3))
unit_sites     = np.zeros((len(xpos)))

for unit in range(len(xpos)):
    cur_xpos = xpos[unit]
    cur_xpos_idx = round(cur_xpos / 250)
    cur_col_idx  = floor((cur_xpos - 250 * cur_xpos_idx) / 32)
    cur_col_idx  = max(0, min(1, cur_col_idx))

    # flip columns if more anterior
    if data_voxels[cur_xpos_idx][0][0][1] < data_voxels[cur_xpos_idx][1][0][1]:
        cur_col_idx = 1 - cur_col_idx

    cur_shank = data_rel_site_locs[cur_xpos_idx]
    cur_col   = cur_shank[cur_col_idx]
    cur_ypos  = ypos[unit]

    closest_col, closest_diff = None, float('inf')
    for col_idx, col_val in enumerate(cur_col[:, 0]):
        diff = abs(col_val - cur_ypos)
        if diff < closest_diff:
            closest_diff = diff
            closest_col = col_idx

    unit_3d_coords[unit] = data_site_locs[cur_xpos_idx][cur_col_idx][closest_col]
    unit_voxels[unit]    = data_voxels[cur_xpos_idx][cur_col_idx][closest_col]
    unit_sites[unit]     = int(data_ccf_sites[cur_xpos_idx][cur_col_idx][closest_col])


3D plot: probes (blue) + mapped units (red)

In [ ]:
fig_plotly = go.Figure()
for probe_idx, probe in enumerate(data_site_locs):
    for col_idx, arr in enumerate(probe):
        fig_plotly.add_trace(go.Scatter3d(
            x=arr[:, 0], y=arr[:, 1], z=arr[:, 2],
            mode='markers', marker=dict(size=3, color='blue'),
            name=f'probe {probe_idx+1} col {col_idx+1}',
        ))
fig_plotly.add_trace(go.Scatter3d(
    x=unit_3d_coords[:, 0], y=unit_3d_coords[:, 1], z=unit_3d_coords[:, 2],
    mode='markers', marker=dict(size=4, color='red'), name='unit_3d_coords'
))
fig_plotly.update_layout(
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
    title='3D positions: probes (blue) and units (red)'
)
pyo.plot(fig_plotly, filename=str(OUT_DIR / '3dpositions_probes_and_units.html'), auto_open=False)


Summary scatter: CCF coords + original xpos/ypos, coloured by site

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.lines import Line2D

fig, ax = plt.subplots(2, 1, figsize=(10, 10))

unique_sites = np.unique(unit_sites)
n_sites = len(unique_sites)
site_to_color_idx = {site: i for i, site in enumerate(unique_sites)}
norm = plt.Normalize(0, n_sites - 1)
cmap = plt.get_cmap('turbo')
color_indices = np.array([site_to_color_idx[s] for s in unit_sites])

ax[0].scatter(unit_3d_coords[:, 1], unit_3d_coords[:, 2], c=color_indices, s=22, cmap='turbo')
ax[0].set_xlabel('ccf Y'); ax[0].set_ylabel('ccf Z')
ax[0].set_xlim([np.max(unit_3d_coords[:, 1]) + 10, np.min(unit_3d_coords[:, 1]) - 20])
ax[0].set_title('3D view of unit_3d_coords')

legend_elements = [
    Line2D([0], [0], marker='o', color='w', label=f'Site {site:.0f}',
           markerfacecolor=cmap(norm(i)), markersize=8)
    for i, site in enumerate(unique_sites)
]
ax[0].legend(handles=legend_elements, title='Unit Sites', loc='upper right')

ax[1].scatter(xpos, ypos, c=color_indices, cmap='turbo', s=4, label='xpos ypos')
for i in [0, 32, 32+32, 250, 282, 282+32, 500, 532, 532+32, 750, 782, 782+32]:
    ax[1].axvline(i)
ax[1].set_xlabel('.mat xpos'); ax[1].set_ylabel('.mat ypos')
ax[1].set_xlim([-100, 1150])
ax[1].legend(handles=legend_elements, title='Unit Sites', loc='upper right')
plt.show()
fig.savefig(str(OUT_DIR / 'unit_sites_3d_and_xpos_ypos.png'), dpi=300, bbox_inches='tight')


Voxel-axis transform: (ML L-R, PA, VD) → (AP, DV, LR), 10 µm units

In [ ]:
unit_voxels[:, 2] = CCF_WIDTH_Z - unit_voxels[:, 2]
unit_voxels[:, 1] = CCF_WIDTH_Y - unit_voxels[:, 1]
unit_voxels = unit_voxels[:, [1, 2, 0]]
unit_voxels = unit_voxels / CCF_RESOLUTION


Save outputs (`.npy` and `.mat`)

In [ ]:
from scipy.io import savemat

OUT_DIR.mkdir(parents=True, exist_ok=True)
np.save(OUT_DIR / 'unit_sites.npy',     unit_sites)
np.save(OUT_DIR / 'unit_3d_coords.npy', unit_3d_coords)
np.save(OUT_DIR / 'unit_voxels.npy',    unit_voxels)
savemat(OUT_DIR / 'unit_sites.mat',     {'unit_sites': unit_sites})
savemat(OUT_DIR / 'unit_3d_coords.mat', {'unit_3d_coords': unit_3d_coords})
savemat(OUT_DIR / 'unit_voxels.mat',    {'unit_voxels': unit_voxels})
print(f"Saved to {OUT_DIR}")


Sanity check: first vs last shank, 3D

In [ ]:
just_first_shanks = np.where(xpos <= 50)[0]
just_last_shanks  = np.where(xpos >= 750)[0]
voxels_first = unit_voxels[just_first_shanks]
voxels_last  = unit_voxels[just_last_shanks]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(voxels_first[:, 0], voxels_first[:, 1], voxels_first[:, 2], c='blue', label='First Shank')
ax.scatter(voxels_last[:, 0],  voxels_last[:, 1],  voxels_last[:, 2],  c='red',  label='Last Shank')
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z'); ax.legend()
plt.title("3D Positions of Units on First and Last Shanks")
plt.show()


---
## Batch runner — discover and process all sessions

Edit `MAT_DIR` / `PKL_ROOT` below, then run the cells to auto-discover sessions by label `YYYYMMDD_MXXX_gN`, pair them with their PKL directory, and run the single-session pipeline for each.

In [ ]:
# --- Batch config ---
from pathlib import Path
import re
import pandas as pd

BATCH_MAT_DIR  = Path("/n/files/Neurobio/MICROSCOPE/Tom/CODE/LC code/processed mat + csv files/29October")
BATCH_PKL_ROOT = Path("/n/files/Neurobio/MICROSCOPE/Tom/CODE/LC code/processed python files/Jul30 herbs to ccf")

label_re = re.compile(r"(\d{8}_M\d+_g\d+)")

def _label_of(p: Path):
    m = label_re.search(p.name)
    return m.group(1) if m else None

# gather xpos/ypos files by label
mats = {}
for p in BATCH_MAT_DIR.glob("*.mat"):
    nm = p.name.lower()
    if ("xpos" not in nm) and ("ypos" not in nm):
        continue
    lab = _label_of(p)
    if not lab:
        continue
    d = mats.setdefault(lab, {"xpos": [], "ypos": []})
    (d["xpos"] if "xpos" in nm else d["ypos"]).append(p)

# for each label, find PKL dir(s) containing the label
rows = []
for lab, dd in sorted(mats.items()):
    n_x, n_y = len(dd["xpos"]), len(dd["ypos"])
    mat_x = dd["xpos"][0] if n_x == 1 else None
    mat_y = dd["ypos"][0] if n_y == 1 else None

    cand_dirs = {p.parent for p in BATCH_PKL_ROOT.rglob(f"*{lab}*/*.pkl")}
    pkl_dir, n_pkl = None, 0
    if cand_dirs:
        counts = [(d_, sum(1 for _ in d_.glob("*.pkl"))) for d_ in cand_dirs]
        counts.sort(key=lambda x: (x[1], str(x[0])), reverse=True)
        pkl_dir, n_pkl = counts[0]

    status_parts = []
    if n_x != 1: status_parts.append(f"xpos={n_x}")
    if n_y != 1: status_parts.append(f"ypos={n_y}")
    if not pkl_dir: status_parts.append("pkl_dir=0")
    status = "OK" if not status_parts else " / ".join(status_parts)

    rows.append({
        "label": lab,
        "mat_x": str(mat_x) if mat_x else "",
        "mat_y": str(mat_y) if mat_y else "",
        "xpos_files_found": n_x,
        "ypos_files_found": n_y,
        "pkl_dir": str(pkl_dir) if pkl_dir else "",
        "n_pkl_files": n_pkl,
        "status": status,
    })

df = pd.DataFrame(rows).sort_values(["status", "label"])
df


In [ ]:
# --- Build SESSIONS from the discovery df ---
def _nskey(p):
    return [int(s) if s.isdigit() else s.lower() for s in re.split(r"(\d+)", p.name)]

ok_rows = df[df["status"] == "OK"].copy()
SESSIONS = []
for _, r in ok_rows.iterrows():
    pdir = Path(r["pkl_dir"])
    pkls = sorted(pdir.glob("*.pkl"), key=_nskey)
    SESSIONS.append(dict(
        tag=r["label"],
        mat_x=str(r["mat_x"]),
        mat_y=str(r["mat_y"]),
        workdir=str(pdir),
        pkl_files=[str(p) for p in pkls],
        flip=False,
    ))

print(f"Built {len(SESSIONS)} sessions.")
if SESSIONS:
    print("Example:", SESSIONS[0]["tag"], "→", len(SESSIONS[0]["pkl_files"]), "PKLs")

missing = df[df["status"].str.contains("pkl_dir=0")]["label"].tolist()
if missing:
    print("No PKL dir found for:", missing)


In [ ]:
# --- run_one: the single-session pipeline as a function ---
def make_out_namer(mat_x_fp, mat_y_fp, tag):
    outdir = Path(mat_x_fp).parent
    outdir.mkdir(parents=True, exist_ok=True)
    def _infer_base():
        for nm in (Path(mat_x_fp).stem, Path(mat_y_fp).stem):
            m = label_re.search(nm)
            if m: return m.group(1)
        sx = re.sub(r'(?i)\b(xpos|ypos)\b', '', Path(mat_x_fp).stem).strip('_-.')
        sy = re.sub(r'(?i)\b(xpos|ypos)\b', '', Path(mat_y_fp).stem).strip('_-.')
        base = sx if len(sx) >= len(sy) else sy
        return base or tag
    base = _infer_base()
    def out(name):
        return str(outdir / f"{base}_{name}")
    return out

def run_one(sess):
    from math import floor
    tag      = sess["tag"]
    workdir  = Path(sess["workdir"])
    mat_x_fp = Path(sess["mat_x"])
    mat_y_fp = Path(sess["mat_y"])
    file_path = [Path(p).name for p in sess["pkl_files"]]
    hemisphere_backwards = bool(sess.get("flip", False))
    out = make_out_namer(mat_x_fp, mat_y_fp, tag)

    xpos = loadmat(str(mat_x_fp))['xpos'].flatten()
    ypos = loadmat(str(mat_y_fp))['ypos'].flatten()

    os.chdir(str(workdir))
    data_site_locs, data_rel_site_locs, data_voxels, data_ccf_sites = [], [], [], []
    missing_key, bad_file = None, None

    for i in file_path:
        with open(i, 'rb') as f:
            try:
                loaded_data = pickle.load(f)
            except ModuleNotFoundError:
                f.seek(0)
                loaded_data = _NPCoreFixUnpickler(f).load()
        d = loaded_data['data']
        rel, used_key = _get_rel(d)
        if rel is None:
            missing_key, bad_file = "sites_loc_relative", i
            break
        if used_key != 'sites_loc_relative':
            print(f"[{tag}] {i}: using '{used_key}' for sites_loc_relative")
        data_site_locs.append(d['sites_loc_b'])
        data_rel_site_locs.append(rel)
        data_voxels.append(d['sites_vox'])
        data_ccf_sites.append(d['sites_label'])

    if missing_key:
        print(f"[SKIP] {tag}: PKL '{bad_file}' lacks {missing_key}.")
        return dict(tag=tag, status='skipped_missing_key', missing=missing_key, file=str(bad_file))

    if hemisphere_backwards:
        for shank in data_site_locs:
            for col in shank:
                col[:, 0] = -col[:, 0]
        for shank in data_voxels:
            for col in shank:
                col[:, 0] = CCF_WIDTH_X - col[:, 0]

    # Normalise relative coords
    for shank in range(len(data_rel_site_locs)):
        for col in range(len(data_rel_site_locs[shank])):
            arr = data_rel_site_locs[shank][col]
            arr[:, 0] -= np.min(arr[:, 0])
            arr[:, 0] = (arr[:, 0] - np.max(arr[:, 0])) * -1
            x_vals = arr[:, 1]
            x_vals[x_vals == 16] = 32 + 250 * shank
            x_vals[x_vals == -8] = 0  + 250 * shank
            arr[:, 1] = x_vals

    valid = np.isfinite(xpos) & np.isfinite(ypos)
    xpos_v, ypos_v = xpos[valid], ypos[valid]
    unit_3d = np.zeros((len(xpos_v), 3))
    unit_vx = np.zeros((len(xpos_v), 3))
    unit_st = np.zeros((len(xpos_v),))
    n_shanks = len(data_rel_site_locs)

    for unit in range(len(xpos_v)):
        cur_xpos = xpos_v[unit]
        cur_xpos_idx = int(np.clip(round(cur_xpos / 250), 0, n_shanks - 1))
        cur_col_idx  = int(np.clip(floor((cur_xpos - 250*cur_xpos_idx) / 32), 0, 1))
        try:
            if (len(data_voxels[cur_xpos_idx]) >= 2
                and len(data_voxels[cur_xpos_idx][0]) > 0
                and len(data_voxels[cur_xpos_idx][1]) > 0
                and data_voxels[cur_xpos_idx][0][0][1] < data_voxels[cur_xpos_idx][1][0][1]):
                cur_col_idx = 1 - cur_col_idx
        except Exception:
            pass
        cur_shank = data_rel_site_locs[cur_xpos_idx]
        if cur_col_idx >= len(cur_shank) or cur_shank[cur_col_idx].size == 0:
            continue
        cur_col = cur_shank[cur_col_idx]
        closest_col, closest_diff = None, float('inf')
        for col_idx, col_val in enumerate(cur_col[:, 0]):
            diff = abs(col_val - ypos_v[unit])
            if diff < closest_diff:
                closest_diff = diff
                closest_col = col_idx
        unit_3d[unit] = data_site_locs[cur_xpos_idx][cur_col_idx][closest_col]
        unit_vx[unit] = data_voxels[cur_xpos_idx][cur_col_idx][closest_col]
        unit_st[unit] = int(data_ccf_sites[cur_xpos_idx][cur_col_idx][closest_col])

    unit_vx[:, 2] = CCF_WIDTH_Z - unit_vx[:, 2]
    unit_vx[:, 1] = CCF_WIDTH_Y - unit_vx[:, 1]
    unit_vx = unit_vx[:, [1, 2, 0]] / CCF_RESOLUTION

    np.save(out('unit_sites.npy'),     unit_st)
    np.save(out('unit_3d_coords.npy'), unit_3d)
    np.save(out('unit_voxels.npy'),    unit_vx)
    savemat(out('unit_sites.mat'),     {'unit_sites': unit_st})
    savemat(out('unit_3d_coords.mat'), {'unit_3d_coords': unit_3d})
    savemat(out('unit_voxels.mat'),    {'unit_voxels': unit_vx})
    print(f"[OK] {tag}: {len(xpos_v)} units → {Path(out('unit_sites.npy')).parent}")
    return dict(tag=tag, status='ok', n_pkls=len(file_path), n_units=len(xpos_v))

# --- Run all sessions ---
summaries = []
for i, s in enumerate(SESSIONS, 1):
    print(f"[{i}/{len(SESSIONS)}] {s['tag']}  @ {s['workdir']}")
    summaries.append(run_one(s))
print("Done.")
summaries
